# Externally calculated metrics

In this notebook, we will be demonstrating how you can upload your own performance metrics to LUSID. This will allow you to use your in-house calculated P&L, Present Values and other metrics.
We will demonstrate this by first creating a portfolio, uploading holdings and then uploading our own performance metrics.

In [ ]:
# Set up LUSID
import os, io
import pandas as pd
import json
import pytz
import matplotlib.pyplot as plt
from IPython.display import HTML, display
from datetime import datetime

import logging
logging.basicConfig(level=logging.INFO)

import lusid as lu
import lusid.api as la
import lusid.models as lm

from lusidjam import RefreshingToken
from lusid.extensions import (
    SyncApiClientFactory,
    ArgsConfigurationLoader,
    EnvironmentVariablesConfigurationLoader,
    SecretsFileConfigurationLoader
)
from finbourne_sdk_utils.cocoon.cocoon import load_from_data_frame
from finbourne_sdk_utils.cocoon import format_instruments_response, format_portfolios_response, format_transactions_response
from finbourne_sdk_utils.jupyter_tools import StopExecution
from finbourne_sdk_utils.lpt.lpt import to_date

# Set pandas display options
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)
pd.options.display.float_format = "{:,.2f}".format

# Set matplotlib display options
%matplotlib inline
plt.style.use('fivethirtyeight')

# Authenticate to SDK
# Run the Notebook in Jupyterhub for your LUSID domain and authenticate automatically
secrets_path = os.getenv("FBN_SECRETS_PATH")
# Run the Notebook locally using a secrets file (see https://support.lusid.com/knowledgebase/article/KA-01663)
if secrets_path is None:
    secrets_path = os.path.join(os.path.dirname(os.getcwd()), "secrets.json")

# Initiate an API Factory which is the client side object for interacting with LUSID APIs
config_loaders=[
    ArgsConfigurationLoader(access_token = RefreshingToken(), app_name = "LusidJupyterNotebook"),
    EnvironmentVariablesConfigurationLoader(),
    SecretsFileConfigurationLoader(secrets_path)]
api_factory = SyncApiClientFactory(config_loaders=config_loaders)

# Confirm success
api_client = api_factory.build(lu.ApplicationMetadataApi)
api_url = api_client.api_client.configuration._base_path.replace("api","")

print ('LUSID Environment :', api_url + "docs")
display(pd.DataFrame(api_client.get_lusid_versions().to_dict()))

# Setup

We must first set up some parameters and initialize the LUSID APIs. Then we will upload the instruments master, create the properties and portfolios and finally upload the holdings.

In [ ]:
scope = "srs-example"

transaction_portfolios_api = api_factory.build(lu.TransactionPortfoliosApi)
srs_api = api_factory.build(lu.StructuredResultDataApi)
instruments_api = api_factory.build(lu.InstrumentsApi)
aggregation_api = api_factory.build(lu.AggregationApi)
configuration_recipe_api = api_factory.build(lu.ConfigurationRecipeApi)
property_definitions_api = api_factory.build(lu.PropertyDefinitionsApi)

## Instruments

We will be upserting the instruments form our instrument master to LUSID so that we may use them later in our example.

In [ ]:
instruments_df = pd.read_csv('data/srs/srs_instrument_master.csv')

result = load_from_data_frame(
    api_factory=api_factory,
    scope=scope,
    data_frame=instruments_df,
    mapping_required={
        "name": "Name"        
    },
    mapping_optional={},
    file_type="instruments",
    identifier_mapping={
        "ClientInternal": "ClientInternal"
    }
)

succ, failed, errors = format_instruments_response(result)
pd.DataFrame(data=[{"success": len(succ), "failed": len(failed), "errors": len(errors)}])

## Properties

We create the following properties as they will serve as sub holding keys. The external results that we will upload can be used on both the holding and sub-holding level within a portfolio.

In [ ]:
def create_property(shk_code, display_name):
    try:
        property_definitions_api.create_property_definition(
            create_property_definition_request=lm.CreatePropertyDefinitionRequest(
                domain="Transaction",
                scope=scope,
                code=shk_code,
                value_required=None,
                display_name=display_name,
                data_type_id=lm.ResourceId(scope="system", code="string"),
                life_time=None,
            )
        )
    except lu.ApiException as e:
        display(json.loads(e.body)["title"])

In [ ]:
create_property("Strategy", "Strategy")
create_property("Sector", "Sector")
create_property("WatchList", "WatchList")

## Portfolios

In [ ]:
portfolio_df = pd.read_csv('data/srs/srs_portfolios.csv')
# Initialize the portfolios API
portfolios_api = api_factory.build(lu.PortfoliosApi)

# Create portfolios directly via the API
for _, row in portfolio_df.iterrows():
    try:
        transaction_portfolios_api.create_portfolio(
            scope=scope,
            create_transaction_portfolio_request=lm.CreateTransactionPortfolioRequest(
                display_name=row['display_name'],
                code=row['code'],
                base_currency=row['base_currency'],
                created="2020-01-01T00:00:00+00:00",
                sub_holding_keys=[
                    f"Transaction/{scope}/Strategy",
                    f"Transaction/{scope}/Sector",
                    f"Transaction/{scope}/WatchList"
                ]
            )
        )
        print(f"Created portfolio {row['code']} successfully")
    except lu.ApiException as e:
        error_response = json.loads(e.body)
        # If portfolio already exists, show a message
        if error_response.get('code') == 409:
            print(f"Portfolio {row['code']} already exists")
        else:
            print(f"Error creating portfolio {row['code']}: {error_response.get('title')}")

## Holdings

We will now upload the holdings, we will do this in multiple lots which we will group by the sub holding keys.

In [ ]:
all_holdings_df = pd.read_csv('data/srs/srs_holdings.csv',parse_dates=['PurchaseDate', 'ValuationDate'])
all_holdings_df.head()

In [ ]:
portfolios = all_holdings_df.groupby(["Portfolio", "ValuationDate"])

for key, holdings_df in portfolios:
    holdings_request = []    
    
    for _, holding in holdings_df.iterrows():
            if not pd.isna(holding["Units"]):
                holdings_request.append(lm.AdjustHoldingRequest(
                    instrument_identifiers={ "Instrument/default/ClientInternal": holding["ClientInternal"] },
                    tax_lots=[lm.TargetTaxLotRequest(
                        units=holding["Units"],
                        cost=lm.CurrencyAndAmount(amount=holding["Cost"], currency=holding["Currency"]),
                        price=holding["Price"],
                        purchase_date=holding["PurchaseDate"]
                    )],
                    sub_holding_keys={
                        f"Transaction/{scope}/Strategy": lm.PerpetualProperty(
                            key=f"Transaction/{scope}/Strategy", value=lm.PropertyValue(label_value=holding["Strategy"])
                        ),
                        f"Transaction/{scope}/Sector": lm.PerpetualProperty(
                            key=f"Transaction/{scope}/Sector", value=lm.PropertyValue(label_value=holding["Sector"])
                        ),
                        f"Transaction/{scope}/WatchList": lm.PerpetualProperty(
                            key=f"Transaction/{scope}/WatchList", value=lm.PropertyValue(label_value=holding["WatchList"])
                        )
                    }
                ))        
        
    print(f"Setting holdings for portfolio {key[0]} at {key[1].isoformat()}")
    transaction_portfolios_api.set_holdings(scope=scope, code=key[0], effective_at=key[1].isoformat(), adjust_holding_request=holdings_request)


# Structured Results Store

Now that we have our portfolios set up, we will demonstrate how you can upsert your own valuation data using the structured results store. We are using the Market Value (MV) and Performance (GainLoss) numbers from our holdings dataset we upserted earlier in the notebook. These values are from one of our externally calculated reports.

In [ ]:
srs_source_df = all_holdings_df[["ValuationDate", "Portfolio", "Strategy", "Sector", "WatchList", "ClientInternal", "Currency", "MV", "GainLoss", "Accrual"]]
srs_source_df

To effectively use the structured results store, we must know the scope and Lusid generated Instrument Ids for each instrument. We can utilize the instruments API to collect this data.

In [ ]:
instruments = instruments_api.get_instruments(identifier_type="ClientInternal", request_body=list(srs_source_df["ClientInternal"].unique()))

In [ ]:
ci_to_luid = {
    ci: inst.identifiers["LusidInstrumentId"]
    for ci, inst in instruments.values.items()
}

srs_source_df["Scope"] = scope
srs_source_df["Luid"] = srs_source_df["ClientInternal"].map(ci_to_luid)
display(srs_source_df)

## Data Map

Now that we have our dataset with our in-house valuations (the MV and GainLoss columns above), we can map these and upload them to Lusid.

In [ ]:
srs_data_map = lm.DataMapping(
    data_definitions=[
        # Here we will define what data and key type each of the data fields are.
        # composite key         
        lm.DataDefinition(address="UnitResult/Portfolio/Id", name="Portfolio", data_type="string", key_type="PartOfUnique"),
        lm.DataDefinition(address="UnitResult/Portfolio/Scope", name="Scope", data_type="string", key_type="PartOfUnique"),
        lm.DataDefinition(address=f"UnitResult/Transaction/{scope}/Strategy", name="Strategy", data_type="string", key_type="PartOfUnique"),
        lm.DataDefinition(address=f"UnitResult/Transaction/{scope}/Sector", name="Sector", data_type="string", key_type="PartOfUnique"),
        lm.DataDefinition(address=f"UnitResult/Transaction/{scope}/WatchList", name="WatchList", data_type="string", key_type="PartOfUnique"),
        lm.DataDefinition(address="UnitResult/Instrument/default/LusidInstrumentId", name="Luid", data_type="string", key_type="PartOfUnique"),
        lm.DataDefinition(address="UnitResult/Holding/default/Currency", name="Currency", data_type="string", key_type="PartOfUnique"),

        # holding values         
        lm.DataDefinition(address="UnitResult/MV", name="MV", data_type="decimal", key_type="Leaf"),
        lm.DataDefinition(address="UnitResult/GainLoss", name="GainLoss", data_type="decimal", key_type="Leaf"),
        lm.DataDefinition(address="UnitResult/Accrual", name="Accrual", data_type="decimal", key_type="Leaf"),
    ]
)

# Because the data maps are immutable, we must increase the version number each time we make any changes and upload a new version
srs_data_map_key = lm.DataMapKey(version="0.1.13", code="market-valuation-map")

# Once the map has been completed, we will create the data map in LUSID
try:    
    srs_api.create_data_map(
        scope=scope, 
        request_body={
            "market-valuation-map": lm.CreateDataMapRequest(
                id=srs_data_map_key,
                data=srs_data_map
            )
        }
    )
except lu.ApiException as e:
    display(json.loads(e.body))


## Upsert Data

In [ ]:
srs_ids = []

for effective_at, srs_df in srs_source_df.groupby("ValuationDate"):

    srs_data_id = lm.StructuredResultDataId(
        source="Client",
        code="MarketValuation",
        effective_at=effective_at.isoformat(),
        result_type = "UnitResult/Holding"
    )
    
    srs_ids.append(srs_data_id)
    
    # filter out rows without any MV and GainLoss values
    srs_df = srs_df[~srs_df[["MV", "GainLoss", "Accrual"]].isna().all(1)]
    s = io.StringIO()
    srs_df.to_csv(s)
    
    srs_data = lm.StructuredResultData(
        document_format="Csv",
        version="0.1.1",
        name="Market valuations",
        data_map_key=srs_data_map_key,
        document=s.getvalue()        
    )
    
    srs_api.upsert_structured_result_data(
        scope=scope, 
        request_body={ 
            "data": lm.UpsertStructuredResultDataRequest(
                id=srs_data_id, 
                data=srs_data
            )
        }
    )

## Read from SRS

We check if everything has been uploaded correctly by retrieving the previously uploaded data.
Here we can see that our upload was successful and the structured results store contains our data.

In [ ]:
for sid in srs_ids:
    
    key = f"{sid.code}-{effective_at}"
    
    values = srs_api.get_structured_result_data(
        scope=scope, 
        request_body={
            key: sid
        }
    )
    
    s = io.StringIO(values.values[key].document)
    values_df = pd.read_csv(s)
    
    display(values_df)

# Valuation Engine
We will now run the valuation engine which will use the structured results store data instead of calculating the valuation data.

## Recipe

In our recipe, we must point out that the recipe should use the data in our structured results store, rather than use the market data in LUSID to calculate the valuations itself.
This happens in the code below, by setting the resource_key parameter to "UnitResult/*".

In [ ]:
recipe_code = "MarketValuation"

recipe = lm.ConfigurationRecipe(
    scope=scope,
    code=recipe_code,
    pricing=lm.PricingContext(
        result_data_rules=[
            lm.ResultDataKeyRule(
                resource_key="UnitResult/*",
                supplier="Client",
                data_scope=scope,
                document_code="MarketValuation",
                quote_interval="1D",
                document_result_type="UnitResult/Holding",
                result_key_rule_type="ResultDataKeyRule",
                useDocumentToInferHoldings=True              # Required to automatically generate zero holdings
            )
        ],
        options = lm.PricingOptions(
            allow_partially_successful_evaluation=True,
            allow_any_instruments_with_sec_uid_to_price_off_lookup=False
        )                              
    )
)

upsert_configuration_recipe_response = configuration_recipe_api.upsert_configuration_recipe(
    upsert_recipe_request = lm.UpsertRecipeRequest(
        configuration_recipe = recipe
    )
)

In [ ]:
def run_valuation(portfolio_code, effective_at):
    
    valuation_request = lm.ValuationRequest(
        recipe_id=lm.ResourceId(scope=scope, code=recipe_code),
        portfolio_entity_ids=[
            lm.PortfolioEntityId(scope=scope, code=portfolio_code)
        ],
        valuation_schedule=lm.ValuationSchedule(effective_at=effective_at.isoformat()),
        metrics=[
            lm.AggregateSpec(key="Portfolio/Scope", op="Value"),
            lm.AggregateSpec(key="Portfolio/Id", op="Value"),
            lm.AggregateSpec(key="Instrument/default/Name", op="Value"),
            lm.AggregateSpec(key=f"Transaction/{scope}/Strategy", op="Value"),
            lm.AggregateSpec(key=f"Transaction/{scope}/Sector", op="Value"),
            lm.AggregateSpec(key=f"Transaction/{scope}/WatchList", op="Value"),
            lm.AggregateSpec(key="Holding/default/Units", op="Value"),
            lm.AggregateSpec(key="UnitResult/MV", op="Value"),
            lm.AggregateSpec(key="UnitResult/GainLoss", op="Value"),
            lm.AggregateSpec(key="UnitResult/Accrual", op="Value"),
        ]
    )
    
    valuation_result = aggregation_api.get_valuation(valuation_request=valuation_request)
    return pd.DataFrame(valuation_result.data)

Now that we have specified the results store data in our recipe and valuation request, we will now run these valuation requests to check if the data returned is indeed the data we uploaded.
We will run these for our 3 dates which we specify below.

In [ ]:
day1 = datetime(2021, 8, 1, tzinfo=pytz.utc)
day2 = datetime(2021, 8, 2, tzinfo=pytz.utc)

Let's first have a look at the EquityA portfolio. Recall from our source file contained in the dataframe "srs_source_df" that we have uploaded the below MV and GainLoss values:

In [ ]:
srs_source_df[srs_source_df['Portfolio'] == "UKEquityA"]

We can now check if we are receiving these same values for UKEquityA from our valuation results:

In [ ]:
run_valuation("UKEquityA", day1)

In [ ]:
run_valuation("UKEquityA", day2)

We can see that these values are the same as from the file we uploaded. We now do the same for the UKEquityB portfolio:

In [ ]:
srs_source_df[srs_source_df['Portfolio'] == "UKEquityB"]

In [ ]:
run_valuation("UKEquityB", day1)

# Note the zero holding for Vodafone has been automatically generated

In [ ]:
run_valuation("UKEquityB", day2)

In [ ]:
# Create links to LUSID Dashboard for viewing our uploaded valuations

# Format dates for URL
date1 = day1.strftime("%Y-%m-%d")
date2 = day2.strftime("%Y-%m-%d")

# Generate URLs for each portfolio and date
portfolio_codes = ["UKEquityA", "UKEquityB"]
dates = [date1, date2]

# Create a HTML table with dashboard links
html_content = "<h3>LUSID Dashboard Valuation Links</h3><table>"
html_content += "<tr><th>Portfolio</th><th>Date</th><th>Link</th></tr>"

for portfolio in portfolio_codes:
    for date in dates:
        dashboard_url = f"{api_url}app/dashboard/valuations/?scope={scope}&code={portfolio}&entityType=Portfolio&recipeScope={scope}&recipeCode={recipe_code}&effectiveDate={date}"
        link_text = f"{portfolio} - {date}"
        html_content += f"<tr><td>{portfolio}</td><td>{date}</td><td><a href='{dashboard_url}' target='_blank'>View in LUSID</a></td></tr>"

html_content += "</table>"

# Display the HTML table with links
display(HTML(html_content))

print("Click on the links above to open the LUSID valuation dashboard for each portfolio and date.")
print(f"The dashboard will use the custom recipe '{recipe_code}' that displays our externally uploaded metrics.")